### Task 5: Model Iterations - Model 1 Logistic Regression

Victoria Vicheva (233182)

Team 9 

### Imports:

In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np
import re
import string
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import pandas as pd
import spacy
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from gensim.models import Word2Vec
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report
import joblib

1. Loading the dataset:

In [14]:
import pandas as pd

# Load each CSV file
df1 = pd.read_csv("goemotions_1.csv")
df2 = pd.read_csv("goemotions_2.csv")
df3 = pd.read_csv("goemotions_3.csv")

# Optionally, inspect the columns to ensure consistency
print("Dataset 1 columns:", df1.columns)
print("Dataset 2 columns:", df2.columns)
print("Dataset 3 columns:", df3.columns)
import pandas as pd

# Load each CSV file
df2 = pd.read_csv("goemotions_2.csv")
df3 = pd.read_csv("goemotions_3.csv")
df1 = pd.read_csv("goemotions_1.csv")

# Combine datasets into one DataFrame
combined_df = pd.concat([df1, df2, df3], ignore_index=True)

# Optionally, remove duplicate rows based on the text column and drop rows with missing text
combined_df.drop_duplicates(subset="text", inplace=True)
combined_df.dropna(subset=["text"], inplace=True)
# Define the mapping from original columns to target emotions
emotion_mapping = {
    "admiration": "happiness",
    "amusement": "happiness",
    "approval": "happiness",
    "caring": "happiness",
    "excitement": "happiness",
    "joy": "happiness",
    "love": "happiness",
    "optimism": "happiness",
    "pride": "happiness",
    "disappointment": "sadness",
    "grief": "sadness",
    "remorse": "sadness",
    "sadness": "sadness",
    "anger": "anger",
    "annoyance": "anger",
    "disapproval": "anger",
    "surprise": "surprise",
    "realization": "surprise",
    "fear": "fear",
    "nervousness": "fear",
    "disgust": "disgust",
    "embarrassment": "disgust",
}

# Include neutral in the target emotions list
target_emotions = [
    "happiness",
    "sadness",
    "anger",
    "surprise",
    "fear",
    "disgust",
    "neutral",
]


def map_row_to_target(row):
    # Initialize cumulative scores for each target emotion
    scores = {emotion: 0 for emotion in target_emotions}
    # Aggregate scores from the mapped emotion columns
    for orig_emotion, target in emotion_mapping.items():
        if orig_emotion in row:
            scores[target] += row[orig_emotion]
    # Also add the score from the 'neutral' column if it exists
    if "neutral" in row:
        scores["neutral"] += row["neutral"]
    # Choose the target emotion with the highest aggregated score
    max_emotion = max(scores, key=scores.get)
    # Optionally, if all scores are zero, you might want to assign 'neutral' or skip the row
    if scores[max_emotion] == 0:
        return None
    return max_emotion


# Apply the mapping function to every row in the combined DataFrame
combined_df["mapped_emotion"] = combined_df.apply(map_row_to_target, axis=1)

# Drop rows where no target emotion could be assigned (if applicable)
combined_df = combined_df.dropna(subset=["mapped_emotion"])
combined_df.to_csv("combined_goemotions_dataset.csv", index=False)
# Load each CSV file
df1 = pd.read_csv("combined_goemotions_dataset.csv")

print("Dataset 1 columns:", df1.columns)

Dataset 1 columns: Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
       'remorse', 'sadness', 'surprise', 'neutral'],
      dtype='object')
Dataset 2 columns: Index(['text', 'id', 'author', 'subreddit', 'link_id', 'parent_id',
       'created_utc', 'rater_id', 'example_very_unclear', 'admiration',
       'amusement', 'anger', 'annoyance', 'approval', 'caring', 'confusion',
       'curiosity', 'desire', 'disappointment', 'disapproval', 'disgust',
       'embarrassment', 'excitement', 'fear', 'gratitude', 'grief', 'joy',
       'love', 'nervousness', 'optimism', 'pride', 'realization', 'relief',
   

In [15]:
import pandas as pd
import spacy
from textblob import TextBlob
from sklearn.feature_extraction.text import TfidfVectorizer
import gensim.downloader as api
from gensim.models import Word2Vec
import numpy as np

# Load your combined dataset
df = pd.read_csv("combined_goemotions_dataset.csv")
df["text"] = df["text"].astype(str)  # Ensure text is a string

############################################
# 1. Part-of-Speech (POS) Tagging with SpaCy
############################################

# Load the English model in SpaCy
nlp = spacy.load("en_core_web_sm")


def get_pos_tags(sentence):
    """Extracts POS tags for a given sentence."""
    doc = nlp(sentence)
    return [token.pos_ for token in doc]


# Create a new column with the POS tags for each sentence
df["POS_Tags"] = df["text"].apply(get_pos_tags)

############################################
# 2. TF-IDF Calculation
############################################

# Initialize a TfidfVectorizer (you can customize tokenization, stop words, etc.)
vectorizer = TfidfVectorizer()
# Fit and transform the text column to obtain the TF-IDF matrix
tfidf_matrix = vectorizer.fit_transform(df["text"])
# Convert the sparse matrix rows to arrays/lists and store in a new column
tfidf_vectors = tfidf_matrix.toarray()
df["TF_IDF"] = list(tfidf_vectors)

############################################
# 3. Sentiment Analysis using TextBlob
############################################


def get_sentiment(text):
    """Returns the sentiment polarity of a sentence."""
    blob = TextBlob(text)
    return blob.sentiment.polarity


# Create a new column with sentiment polarity scores for each sentence
df["Sentiment_Score"] = df["text"].apply(get_sentiment)

############################################
# 4. Pretrained Word Embeddings (GloVe)
############################################

# Load a pretrained embedding model from Gensim (e.g., 'glove-wiki-gigaword-100')
pretrained_model = api.load("glove-wiki-gigaword-100")  # 100-dimensional vectors


def get_pretrained_embedding(sentence):
    """
    Compute the average pretrained embedding for a sentence.
    Only words present in the pretrained model are considered.
    """
    doc = nlp(sentence)
    tokens = [
        token.text.lower() for token in doc if not token.is_punct and not token.is_space
    ]
    vectors = [pretrained_model[word] for word in tokens if word in pretrained_model]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        # If no word vectors found, return a zero vector
        return np.zeros(pretrained_model.vector_size)


# Create a new column with the average pretrained word embedding per sentence
df["Pretrained_Embeddings"] = df["text"].apply(get_pretrained_embedding)

############################################
# 5. Custom Word Embedding Model with Word2Vec
############################################

# First, tokenize each sentence using SpaCy (reuse nlp from above)
df["tokens"] = df["text"].apply(
    lambda x: [
        token.text.lower()
        for token in nlp(x)
        if not token.is_punct and not token.is_space
    ]
)
# Prepare sentences for training the custom Word2Vec model
sentences = df["tokens"].tolist()

# Train a Word2Vec model on your dataset tokens (tweak vector_size, window, etc. as needed)
custom_model = Word2Vec(sentences, vector_size=100, window=5, min_count=1, workers=4)


def get_custom_embedding(token_list):
    """
    Compute the average custom embedding for a list of tokens from a sentence.
    Only tokens present in the custom model are considered.
    """
    vectors = [custom_model.wv[word] for word in token_list if word in custom_model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(custom_model.vector_size)


# Create a new column with the custom embeddings per sentence
df["Custom_Embeddings"] = df["tokens"].apply(get_custom_embedding)

############################################
#  Save the Enhanced Dataset
############################################

df.to_csv("enhanced_goemotions_dataset_with_features.csv", index=False)

In [16]:
import spacy
import numpy as np
import pandas as pd

# Load the spaCy model with word vectors.
nlp = spacy.load("en_core_web_sm")


def extract_pretrained_embedding(sentence):
    """
    Given a sentence, this function extracts token vectors using spaCy
    and returns the average vector as a list.
    The "en_core_web_md" model provides 300-dimensional vectors.
    """
    doc = nlp(sentence)
    # Collect vectors for tokens that have vectors.
    vectors = [token.vector for token in doc if token.has_vector]
    if vectors:
        avg_vector = np.mean(vectors, axis=0)
        return avg_vector.tolist()  # Convert numpy array to list
    else:
        # Return a zero vector if no token has a vector.
        return [0.0] * 300


# ---------------------------
# Step 1. Load Existing Data
# ---------------------------
existing_df = pd.read_csv("enhanced_goemotions_dataset_with_features.csv")


# ---------------------------
# Step 2. Load New Data
# ---------------------------
new_df = pd.read_csv("group 9_url1.csv")


# ---------------------------
# Step 3. Extract Additional NLP Feature from New Data
# ---------------------------
# Here, we choose the "Corrected Sentence" column for extraction.
new_df["Pretrained_Embeddings"] = new_df["Corrected Sentence"].apply(
    extract_pretrained_embedding
)


# For label mapping (if applicable), assume new_df has an "Emotion" column.
label_to_id = {
    "happiness": 0,
    "sadness": 1,
    "anger": 2,
    "surprise": 3,
    "fear": 4,
    "disgust": 5,
    "neutral": 6,
}
if "Emotion" in new_df.columns:
    new_df["label_id"] = new_df["Emotion"].map(label_to_id)
else:
    new_df["label_id"] = None  # or handle as needed

# ---------------------------
# Step 4. Combine the Data
# ---------------------------

new_df.rename(columns={"Corrected Sentence": "text"}, inplace=True)


combined_df = pd.concat([existing_df, new_df], ignore_index=True)
print("Combined data shape:", combined_df.shape)

# ---------------------------
# Step 5. Save or Continue with Training
# ---------------------------
# Save the combined data to a new CSV file for consistency.
combined_df.to_csv("combined_data_with_features.csv", index=False)

print(
    "Data processing complete. The combined data now includes the additional NLP feature."
)

Combined data shape: (51869, 51)
Data processing complete. The combined data now includes the additional NLP feature.


### Iteration 1: 

Features Used: TF-IDF

#### Loading the combined and enhanced dataset:

In [17]:
import pandas as pd

# Load the dataset
df = pd.read_csv("combined_data_with_features.csv")

# Ensure text is a string
df["text"] = df["text"].astype(str)

# Mapping emotions to numerical labels
label_to_id = {
    "happiness": 0,
    "sadness": 1,
    "anger": 2,
    "surprise": 3,
    "fear": 4,
    "disgust": 5,
    "neutral": 6,
}
df["label"] = df["mapped_emotion"].map(label_to_id)

# Drop missing labels
df = df.dropna(subset=["label"])

d:\anaconda\envs\block_c_y2\lib\site-packages\IPython\core\interactiveshell.py:3508: DtypeWarning: Columns (8,44,45,46,47,48,49) have mixed types.Specify dtype option on import or set low_memory=False.
  exec(code_obj, self.user_global_ns, self.user_ns)


#### Splitting data into training and val sets

splitting the dataset into 80% training and 20% val.

In [18]:
from sklearn.model_selection import train_test_split

# Split 80% train, 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    df["text"], df["label"], test_size=0.2, random_state=42, stratify=df["label"]
)

#### Training the Naive Bayes model

Multinomial Naive Bayes will be used, which is commonly used for text classification.

In [19]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import make_pipeline

# Create a TF-IDF + Naïve Bayes pipeline
model = make_pipeline(TfidfVectorizer(), MultinomialNB())

# Train the model
model.fit(X_train, y_train)

Pipeline(steps=[('tfidfvectorizer', TfidfVectorizer()),
                ('multinomialnb', MultinomialNB())])

#### Evaluation of the model

In [20]:
from sklearn.metrics import accuracy_score, classification_report

# Make predictions
y_pred = model.predict(X_val)

# Evaluate performance
accuracy = accuracy_score(y_val, y_pred)
report = classification_report(y_val, y_pred, target_names=label_to_id.keys())

# Print results
print(f"Accuracy: {accuracy:.4f}")
print(report)

Accuracy: 0.4451
              precision    recall  f1-score   support

   happiness       0.44      0.91      0.60      3920
     sadness       1.00      0.00      0.01       781
       anger       0.88      0.02      0.03      1412
    surprise       0.00      0.00      0.00       526
        fear       0.00      0.00      0.00       178
     disgust       0.00      0.00      0.00       229
     neutral       0.45      0.30      0.36      3094

    accuracy                           0.45     10140
   macro avg       0.40      0.18      0.14     10140
weighted avg       0.51      0.45      0.34     10140



d:\anaconda\envs\block_c_y2\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\anaconda\envs\block_c_y2\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
d:\anaconda\envs\block_c_y2\lib\site-packages\sklearn\metrics\_classification.py:1318: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
